# Generate Patch Comparisons
Extracts real 64×64 patches from the ARCADE validation set, runs inference, and creates side-by-side comparison images.

**Requirements on Drive:**
- `MyDrive/CMT/runs/<run_id>/best.pth` — trained checkpoint
- `MyDrive/arcade.zip` — ARCADE dataset

In [ ]:
# 1. Setup
!git clone https://github.com/C0d3Crush/xray-vessel-inpainting.git 2>/dev/null || (cd xray-vessel-inpainting && git pull)
%cd xray-vessel-inpainting
!pip install -q torch torchvision matplotlib pillow opencv-python scikit-image pycocotools pandas numpy scipy
print('Setup complete')

In [ ]:
# 2. Mount Drive and extract dataset
from google.colab import drive
import os, glob

drive.mount('/content/drive')

ARCADE_ZIP = '/content/drive/MyDrive/arcade.zip'
!unzip -q -o "$ARCADE_ZIP" -d /content/xray-vessel-inpainting/data/

# Auto-detect val annotation + image paths
import json
from collections import defaultdict

STENOSIS_CAT = 26

def count_usable(ann_path):
    with open(ann_path) as f:
        coco = json.load(f)
    by_img = defaultdict(list)
    for a in coco.get('annotations', []):
        if a['category_id'] != STENOSIS_CAT:
            by_img[a['image_id']].append(a)
    return sum(1 for img in coco.get('images', []) if by_img[img['id']])

candidates = []
for match in glob.glob('data/**/val.json', recursive=True):
    img_dir = os.path.join(os.path.dirname(os.path.dirname(match)), 'images')
    if os.path.isdir(img_dir):
        n = count_usable(match)
        candidates.append((match, img_dir, n))

candidates.sort(key=lambda c: -c[2])
if not candidates:
    raise RuntimeError('No val.json found — check that arcade.zip extracted correctly')

VAL_ANN, VAL_IMG, val_n = candidates[0]
print(f'Val annotations: {VAL_ANN}  ({val_n} usable images)')
print(f'Val images:      {VAL_IMG}')

In [ ]:
# 3. Select checkpoint from Drive
import glob, os

# List all available runs on Drive
runs = sorted(glob.glob('/content/drive/MyDrive/CMT/runs/*/best.pth'))
if not runs:
    # Fallback: check legacy location
    runs = sorted(glob.glob('/content/drive/MyDrive/CMT/checkpoints/best.pth'))

print('Available checkpoints:')
for i, r in enumerate(runs):
    size_mb = os.path.getsize(r) / 1e6
    print(f'  [{i}] {r}  ({size_mb:.1f} MB)')

# Change this index to pick a different run
CKPT_IDX = -1  # -1 = latest run
CKPT_PATH = runs[CKPT_IDX]
print(f'\nUsing: {CKPT_PATH}')

In [ ]:
# 4. Config — adjust these to change the output
NUM_IMAGES       = 6    # how many full images to sample patches from
PATCHES_PER_IMG  = 2    # patches extracted per image
PATCH_SIZE       = 64
VESSEL_BIAS      = 0.8  # probability of centering patch on a vessel pixel
MASK_PADDING     = 0    # dilation radius around vessel annotations (px)
SEED             = 42

OUTPUT_DIR = 'outputs/comparisons'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Config OK — will generate {NUM_IMAGES * PATCHES_PER_IMG} patches total')

In [ ]:
# 5. Run patch inference
import subprocess, sys

PATCH_DIR = os.path.join(OUTPUT_DIR, 'patches')

cmd = [
    sys.executable, 'scripts/patch_inference.py',
    '--ckpt',             CKPT_PATH,
    '--annotations',      VAL_ANN,
    '--images',           VAL_IMG,
    '--output-dir',       PATCH_DIR,
    '--num-images',       str(NUM_IMAGES),
    '--patches-per-image',str(PATCHES_PER_IMG),
    '--patch-size',       str(PATCH_SIZE),
    '--vessel-bias',      str(VESSEL_BIAS),
    '--mask-padding',     str(MASK_PADDING),
    '--seed',             str(SEED),
    '--no-comparison',    # cell 6 creates the timestamped comparison instead
    '--device',           'cuda' if __import__('torch').cuda.is_available() else 'cpu',
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# 6. Create comparison image
from datetime import datetime

run_label = os.path.basename(os.path.dirname(CKPT_PATH))
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
COMPARISON_PATH = os.path.join(OUTPUT_DIR, f'comparison_{run_label}_{timestamp}.png')

cmd = [
    sys.executable, 'scripts/create_training_comparison.py',
    '--patch-img',    os.path.join(PATCH_DIR, 'original'),
    '--patch-mask',   os.path.join(PATCH_DIR, 'mask'),
    '--patch-result', os.path.join(PATCH_DIR, 'result'),
    '--output',       COMPARISON_PATH,
    '--title',        f'Real 64×64 Patch Inference — run {run_label}',
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# 7. Display result
from IPython.display import Image, display
import os

if os.path.exists(COMPARISON_PATH):
    display(Image(COMPARISON_PATH))
else:
    print(f'File not found: {COMPARISON_PATH}')

In [ ]:
# 8. Save to Drive
import shutil

DRIVE_DEST = f'/content/drive/MyDrive/CMT/comparisons/'
os.makedirs(DRIVE_DEST, exist_ok=True)

dest = shutil.copy(COMPARISON_PATH, DRIVE_DEST)
print(f'Saved to Drive: {dest}')